In [1]:
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence
from gensim.models.callbacks import CallbackAny2Vec

sentences = LineSentence("/kaggle/input/listenbrainz-sentences/sentences.txt")

class CheckpointCallback(CallbackAny2Vec):
    def __init__(self, save_path):
        self.save_path = save_path
        self.epoch = 0

    def on_epoch_end(self, model):
        self.epoch += 1
        path = f"{self.save_path}_epoch{self.epoch}.model"
        model.save(path)
        print(f"Checkpoint saved: {path}")

checkpoint = CheckpointCallback("item2vec_checkpoint")

model = Word2Vec(
    sentences=sentences,
    vector_size=128,
    window=10,
    min_count=20,        # songs seen fewer than 20 times get no vector
    sg=1,               # skip-gram
    negative=10,
    sample=1e-4,
    workers=4,
    epochs=5,
    callbacks=[checkpoint],
)

model.save("item2vec.model")
print("Training complete")
print(f"Vocabulary size: {len(model.wv)}")

Checkpoint saved: item2vec_checkpoint_epoch1.model
Checkpoint saved: item2vec_checkpoint_epoch2.model
Checkpoint saved: item2vec_checkpoint_epoch3.model
Checkpoint saved: item2vec_checkpoint_epoch4.model
Checkpoint saved: item2vec_checkpoint_epoch5.model
Training complete
Vocabulary size: 802724


In [2]:
# Pick a random word from the vocabulary and find its nearest neighbors
import random
test_mbid = random.choice(model.wv.index_to_key)

print(f"Query MBID: {test_mbid}")
print(f"\n10 nearest neighbors:")
for mbid, score in model.wv.most_similar(test_mbid, topn=10):
    print(f"  {score:.4f}  {mbid}")

Query MBID: 23df4c5a-258b-426c-8e69-afc8fccd1db2

10 nearest neighbors:
  0.9127  a63006db-dd6b-4329-8641-38cc9faaded2
  0.9031  2d550e63-097d-4cef-ae5e-65bf575fab3b
  0.9022  92db0e73-e67a-4e51-9d76-23050e4a0030
  0.9003  acaa82f8-c4ba-4658-bfde-2ba57fe913a8
  0.8998  786435c1-e1e9-4487-b0ca-8671e3dea1cb
  0.8974  aa0f69ec-6af7-4eb7-b5ce-b7176af36388
  0.8946  3d39310a-82d3-4799-b0b8-b84dd4739925
  0.8939  ca547253-f15c-4651-8e82-91bd2a14815b
  0.8935  1a4373e0-9a37-4b17-8b2b-6d0e6fd6cd2b
  0.8929  b93d7a4a-9e89-4405-bb93-8bd64ca0188f


In [3]:
import numpy as np

# Export vectors only — this is what the backend will actually load
# keyed_vectors can be loaded without the full model
model.wv.save("item2vec.wordvectors")

# Also export as numpy for inspection and for building the Annoy index
mbids = model.wv.index_to_key
vectors = model.wv.vectors

np.save("mbids.npy", np.array(mbids))
np.save("vectors.npy", vectors)

print(f"Exported {len(mbids)} vectors of dimension {vectors.shape[1]}")

Exported 802724 vectors of dimension 128


In [4]:
import subprocess
subprocess.run(["pip", "install", "annoy"], check=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 156.2 kB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for annoy: filename=annoy-1.17.3-cp312-cp312-linux_x86_64.whl size=551808 sha256=d531d37e8ac063a1ad24bc3b573cafc1f9dc2d94a2494a70fe51bf3154a2a8dd
  Stored in directory: /root/.cache/pip/wheels/db/b9/53/a3b2d1fe1743abadddec6aa541294b24fdbc39d7800bc57311
Successfully built annoy


CompletedProcess(args=['pip', 'install', 'annoy'], returncode=0)

In [5]:
from annoy import AnnoyIndex

dims = model.wv.vector_size
index = AnnoyIndex(dims, "angular")

for i, mbid in enumerate(model.wv.index_to_key):
    index.add_item(i, model.wv[mbid])

index.build(50)  # 50 trees — more trees = more accurate but slower to build
index.save("song_index.ann")

print(f"Annoy index built with {len(model.wv)} items")

Annoy index built with 802724 items


In [6]:
import json

mbid_to_idx = {mbid: i for i, mbid in enumerate(model.wv.index_to_key)}
idx_to_mbid = {i: mbid for mbid, i in mbid_to_idx.items()}

with open("mbid_to_idx.json", "w") as f:
    json.dump(mbid_to_idx, f)

with open("idx_to_mbid.json", "w") as f:
    json.dump(idx_to_mbid, f)

print("Mappings saved")

Mappings saved
